Importing all the essential libraries

In [73]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import stopwords
import re
from collections import defaultdict
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

In [ ]:
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

Creating function to preprocess the text

In [75]:
def preprocess_text(text):
    text=text.lower()
    text=re.sub(r'[^\w\s]','',text)
    sentences=sent_tokenize(text)
    return sentences

Creating function to gather proofs or basis in order to decide the task by coming up with an approach of verbs and time.

In [76]:
def identify_task():

    task_hints={
        'verbs':{'must','should','have to','need to','has to','had to'},
        'action_verbs':{'complete', 'finish', 'submit', 'prepare', 'review', 
                        'update', 'create', 'develop', 'implement', 'clean', 
                        'buy', 'get', 'send', 'write', 'call'},
        'time':{'by','before','after','when','while','during','until','as soon as','as long as','deadline'},
    }
    
    return task_hints

Creating function to extract the time or date from the text to associate it with extracted task logically.

In [77]:
def extract_time(sentence):
    time_patterns=[
        r'by\s+(.*?)(\.|\s|$)',
        r'before\s+(.*?)(\.|\s|$)',
        r'due\s+(.*?)(\.|\s|$)',
        r'until\s+(.*?)(\.|\s|$)'
    ]
    for pattern in time_patterns:
        match = re.search(pattern, sentence)
        if match:
            return match.group(1).strip()
    return None

Creating a function to get the name or the doer of the task.

In [78]:
def extract_person(sentence):
    words=word_tokenize(sentence)
    tagged_words=pos_tag(words)
    for word,tag in tagged_words:
        if tag.startswith('NNP') and word[0].issupper():
            return word
    return None

Creating a function to use the basis extracted before to extract the tasks.

In [79]:
def is_task(sentence,task_hints):
    words=word_tokenize(sentence)
    for word in words:
        if word in task_hints['verbs']:
            return True
        if word in task_hints['action_verbs']:
            return True
    if len(words) > 0:
        tagged_words = pos_tag(words)
        if tagged_words[0][1].startswith('VB'):
            return True
    return False

Creating a function to categorize the task that is extracted.

In [80]:
def categorize(tasks, n_categories=5):
    if not tasks:
        return []
    vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
    tfidf_matrix = vectorizer.fit_transform([task['task'] for task in tasks])

    n_categories=min(n_categories,len(tasks))

    Kmeans=KMeans(n_clusters=n_categories,random_state=42)
    clusters=Kmeans.fit_predict(tfidf_matrix)

    feature_name=vectorizer.get_feature_names_out()
    cluster_centers=Kmeans.cluster_centers_

    categories=[]
    for center in cluster_centers:
        top_terms_idx=center.argsort()[-2:][::-1]
        category='&'.join(feature_name[i] for i in top_terms_idx)
        categories.append(category)

    return [categories[cluster] for cluster in clusters]

Finally a functio that call all the previous functions in order to get the task, assignee and deadline, moreover formating and returning it in a user understandble way.

In [88]:
def extract_tasks(text):

    indicators=identify_task()
    t=preprocess_text(text)
    sentences = re.split(r'\n+|(?<=\.) ', " ".join(t).strip())

    tasks=[]
    for sentence in sentences:
        if is_task(sentence,indicators):
            task_info={
                'task':sentence.strip().replace("\n"," "),
                'assignee':extract_person(sentence),
                'deadline':extract_time(sentence)
            }
            tasks.append(task_info)

    if tasks:
        df=pd.DataFrame(tasks)
        df['category']=categorize(tasks)
        return df
    return pd.DataFrame()
    

In [ ]:
test_text = """
    Rahul wakes up early every day.
    He goes to college in the morning and comes back at 3 pm. 
    At present, Rahul is outside. 
    He has to buy the snacks for all of us.
    """

results = extract_tasks(test_text)
print("\nExtracted Tasks:")
print(results)


Extracted Tasks:
                                     task assignee deadline    category
0  he has to buy the snacks for all of us     None     None  snacks&buy
